In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.6.0+cu126.html
!pip install torch_geometric sparse
!pip install pycocoevalcap h5py tensorboardX boto3 requests pandas tqdm nltk ftfy open_clip_torch transformers
!pip install stanfordcorenlp

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('words')

import json
from tqdm import tqdm
from stanfordcorenlp import StanfordCoreNLP
import re
import string
import pandas as pd
import json
from nltk.corpus import stopwords
from nltk.corpus import words

In [ ]:
!wget http://nlp.stanford.edu/software/stanford-corenlp-4.5.4.zip
!unzip stanford-corenlp-4.5.4.zip -d /content/
print("Stanford CoreNLP downloaded and extracted.")

In [ ]:
import os

os.chdir("/content/drive/MyDrive/0_Note_KLTN/Test_code/video_datasets/KG")

## I. Get Sentences

### 1. MSRVTT

In [ ]:
# @title
dataset = 'msrvtt'
# dataset = 'msvd'
# dataset = 'didemo'

def msvd_load_captions(caption_fpath):
    df = pd.read_csv(caption_fpath)
    df = df[df['Language'] == 'English']
    df = df[pd.notnull(df['Description'])]
    captions = df['Description'].values
    return captions

if dataset == 'msrvtt':
    file_path = './MSRVTT/metadata/MSRVTT_data_train.json'
    sentence_path = './MSRVTT/metadata/msrvtt_sentence.txt'

    cnt = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        load_data = json.load(f)

    with open(sentence_path, 'w', encoding='utf-8') as w:
        for sample in load_data:
            captions = sample['caption']

            for caption in captions:
                w.write(caption + '\n')
                print(caption)
                cnt += 1

    print(cnt)

if dataset == 'msvd':
    cnt = 0
    train_id_path = './MSVD/metadata/train.list'
    train_sentence_path = './MSVD/metadata/train.csv'
    sentence_path = './MSVD/metadata/msvd_sentence.txt'
    w = open(sentence_path, 'w')
    sentence = msvd_load_captions(train_sentence_path)
    # print(sentence.size)
    for i in range(sentence.size):
        w.writelines(sentence[i])
        w.write('\n')
        cnt += 1
    w.close()
    print(cnt)

if dataset == 'didemo':
    file_path = './DiDeMo/metadata/train_data.json'
    sentence_path = './DiDeMo/metadata/didemo_sentence.txt'
    cnt = 0

    with open(file_path, 'r') as f:
        load_data = json.load(f)

    w = open(sentence_path, 'w')

    for item in load_data:
        caption = item['description']

        w.writelines(caption)
        w.write('\n')

        cnt += 1
        print(caption)

    w.close()
    print(cnt)

## II. Extract triples from sentences using Standford NLP

In [ ]:
nlp = StanfordCoreNLP('/content/stanford-corenlp-4.5.4', lang='en')

dataset = 'msrvtt'
# dataset = 'msvd'
# dataset = 'didemo'

if dataset == 'msvd':
    sentence_path = './MSVD/metadata/msvd_sentence.txt'
elif dataset == 'msrvtt':
    sentence_path = './MSRVTT/metadata/msrvtt_sentence.txt'
elif dataset == 'didemo':
    sentence_path = './DiDeMo/metadata/didemo_sentence.txt'

entity_rel_path = f"{dataset}_entity_total.txt" # Output to current working directory with clearer name
log_path = 'error_log.txt'

f = open(sentence_path, 'r')
w = open(entity_rel_path, 'w')
log = open(log_path, 'w')
lines = f.readlines()
cnt = 0
num_rel = 0

for line in tqdm(lines):
    sentence = line.strip()
    try:
        output = nlp.annotate(sentence, properties={
            "annotators": "tokenize,lemma,ssplit,pos,depparse,natlog,openie",
            "outputFormat": "json",
            'openie.triple.strict': 'true',
            'openie.max_entailments_per_clause': '1'
        })

        data = json.loads(output)
        for i in range(len(data['sentences'])):
            result = [data["sentences"][i]["openie"]]
            lemmas = data["sentences"][i]["tokens"]
            cnt += 1

            for g in result:
                for rel in g:
                    subj_start, subj_end = rel['subjectSpan']
                    obj_start, obj_end = rel['objectSpan']
                    rel_start, rel_end = rel['relationSpan']

                    if max(subj_end, obj_end, rel_end) > len(lemmas):
                        log.write(f"[Warning] Span out of range at sentence {cnt}: {sentence}\n")
                        continue

                    # Sử dụng join thay vì nối chuỗi ngược
                    l_subject = ' '.join([token['lemma'] for token in lemmas[subj_start:subj_end]])
                    l_object = ' '.join([token['lemma'] for token in lemmas[obj_start:obj_end]])
                    # l_relation = ' '.join([token['lemma'] for token in lemmas[rel_start:rel_end]])

                    subj_tokens = [
                        token['lemma'] for token in lemmas[subj_start:subj_end]
                        if token['pos'].startswith('NN')
                    ]
                    l_subject = ' '.join(subj_tokens)

                    obj_tokens = [
                        token['lemma'] for token in lemmas[obj_start:obj_end]
                        if token['pos'].startswith('NN')
                    ]
                    l_object = ' '.join(obj_tokens)

                    relation_tokens = [
                        token['lemma'] for token in lemmas[rel_start:rel_end]
                        if token['pos'].startswith('VB')
                    ]
                    l_relation = ' '.join(relation_tokens)

                    if not l_subject or not l_object or not l_relation:
                        continue

                    relationSent = f"{l_subject.strip()} &{l_object.strip()} &{l_relation.strip()}"
                    w.write(relationSent + '\n')
                    num_rel += 1

    except Exception as e:
        log.write(f"[Error] Exception at sentence {cnt}: {sentence}\n")
        log.write(f"         Error message: {str(e)}\n")
w.close()
f.close()
log.close()
nlp.close()

In [ ]:
print('Result written to:', entity_rel_path)
print('Error log written to:', log_path)
print('Total number of processed sentences: ' + str(cnt))
print('Total number of relations extracted: ' + str(num_rel))

## III. Clean entities

In [ ]:
dataset = 'msrvtt'
# dataset = 'msvd'
# dataset = 'didemo'

In [ ]:
# h - t - r
english_vocab = set(words.words())
stop_words = set(stopwords.words('english'))
akg_vocab = set()

def load_triples(filepath):
    triples = []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split('&')
            if len(parts) == 3:
                head, tail, relation = parts
                triples.append((head.strip(), tail.strip(), relation.strip()))
    return triples

def is_valid_entity(entity):
    tokens = entity.split()
    for token in tokens:
        if token in akg_vocab:
            continue
        if len(token) == 1 or not token.isalpha() or token not in english_vocab:
            return False
    return True

def filter_triples(triples):
    remove_relations = {"be", "have", "do", "get", "can", "is", "are"}
    filtered = []
    for h, t, r in triples:   # FIX
        if r in remove_relations:
            continue
        if len(h.split()) < 1 or len(t.split()) < 1:
            continue
        if len(h.split()) > 2 or len(t.split()) > 2:
            continue
        if not is_valid_entity(h) or not is_valid_entity(t):
            continue
        filtered.append((h, t, r))   # FIX
    return filtered

def normalize_triples(triples):
    def normalize_text(text):
        quantifiers = {
            "some", "many", "several", "few", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
            "all", "both", "each", "every", "no", "none", "any", "most", "more", "less"
        }
        quantifiers.update(str(i) for i in range(1, 51))
        text = text.lower()
        text = re.sub(rf"[{string.punctuation}]", "", text)
        tokens = text.strip().split()
        tokens = [w for w in tokens if w not in stop_words.union(quantifiers)]
        return ' '.join(tokens)

    norm_triples = []
    for h, t, r in triples:   # FIX
        norm_h = normalize_text(h)
        norm_t = normalize_text(t)
        norm_r = normalize_text(r)
        if norm_h and norm_t and norm_r:
            norm_triples.append((norm_h, norm_t, norm_r))  # FIX
    return norm_triples

triples = load_triples(dataset + "_entity_total.txt")
triples = normalize_triples(triples)
triples = filter_triples(triples)

with open(dataset + "_cleaned_entities.txt", "w") as f:
    for h, t, r in triples:
        relationSent = f"{h} &{t} &{r}"
        f.write(relationSent + '\n')

print('number of triples:', len(triples))

## IV. Filter triplets

In [ ]:
dataset = 'msrvtt'
# dataset = 'msvd'
# dataset = 'didemo'

In [ ]:
def deduplicate_triples(triples):
    return list(set(triples))

triples = load_triples(dataset + "_cleaned_entities.txt")
# triples = normalize_triples(triples)
# triples = filter_triples(triples)
triples = deduplicate_triples(triples)

with open(dataset + "_filtered_entities.txt", "w") as f:
    for h, t, r in triples:
        relationSent = f"{h} &{t} &{r}"
        f.write(relationSent + '\n')
print('number of triples:', len(triples))